In [1]:
import os

from pathlib import Path
from typing import (
    Optional, 
    Dict,
    Any,
    List, 
    Tuple,
)

import pandas as pd
import xarray as xr
import numpy  as np

from tqdm import tqdm
from logger import get_logger
from scipy import stats as scipy_stats
from concurrent.futures import ProcessPoolExecutor, as_completed

from constants import (
    BARRA_CATALOGUE,
    EAR5_CATALOGUE,
    ERA5L_CATALOGUE,
    CREATEIP_CATALOGUE,
    BARRA_VARIABLES
)

logger = get_logger()


QUANTILES = [
    0.01, 0.02, 0.03, 0.04, 0.05,
    0.25, 0.50, 0.75,
    0.95, 0.96, 0.97, 0.98, 0.99
]
MAX_WORKERS = 48


In [2]:
%%time
BARRA = pd.read_csv(BARRA_CATALOGUE, compression='gzip')

BARRA = BARRA[BARRA.source_id == 'BARRA-RE2']
BARRA = BARRA[BARRA.domain_id == 'AUS-22']
BARRA = BARRA[BARRA.freq == 'day']
BARRA = BARRA[BARRA.file_type == 'f']
BARRA = BARRA[BARRA.variable_id.isin(BARRA_VARIABLES)]

# Convert start_time safely: 200501.0 -> 200501 -> datetime
BARRA['start_time'] = pd.to_numeric(
    BARRA['start_time'],
    errors='coerce'
).astype('Int64')

BARRA['start_time'] = pd.to_datetime(
    BARRA['start_time'].astype(str),
    format='%Y%m',
    errors='coerce'
)

# Remove failed dates, if any
BARRA = BARRA.dropna(subset=['start_time'])

# Select last 20 calendar years available
latest_year = BARRA['start_time'].dt.year.max()
start_year = latest_year - 35

BARRA = BARRA[
    BARRA['start_time'].dt.year.between(start_year, latest_year)
]

# Convert back to YYYY-MM
BARRA['start_time'] = BARRA['start_time'].dt.strftime('%Y-%m')

BARRA = (
    BARRA[[
    'variable_id',
    'start_time',
    'path'
    ]]
    .sort_values(by='start_time')
    .reset_index(drop=True)
)

CPU times: user 8.41 s, sys: 627 ms, total: 9.03 s
Wall time: 9.03 s


In [3]:
def extract_metadata_from_netcdf(file_path):
    try:
        with xr.open_dataset(file_path, decode_times=True, chunks={}) as ds:
            dates = []

            if 'time' in ds:
                time_values = ds['time'].values
                dates = [
                    pd.Timestamp(t).strftime('%Y-%m-%d')
                    for t in time_values
                ]

            data_vars = [v for v in ds.data_vars if v not in ds.coords]

            if data_vars:
                var_name = data_vars[0]
                dims = {
                    k: v
                    for k, v in ds[var_name].sizes.items()
                    if k != 'time'
                }
                dimensions = str(dims)
            else:
                dimensions = None

            return {
                'path': file_path,
                'dates': dates,
                'dimensions': dimensions
            }

    except Exception as e:
        return {
            'path': file_path,
            'dates': [],
            'dimensions': None,
            'error': str(e)
        }


def extract_dates_concurrent(
    catalogue: pd.DataFrame,
    max_workers= None,
) -> pd.DataFrame:

    if max_workers is None:
        max_workers = min(8, os.cpu_count() or 1)

    catalogue = catalogue.copy()

    # Remove old date columns if present
    cols_to_drop = ['start_date', 'end_date', 'date', 'dimensions']
    catalogue = catalogue.drop(
        columns=[c for c in cols_to_drop if c in catalogue.columns]
    )

    if 'path' not in catalogue.columns:
        raise ValueError("catalogue must contain a 'path' column")

    file_paths = catalogue['path'].dropna().unique().tolist()
    total_files = len(file_paths)

    logger.info(
        f"Extracting metadata from {total_files} NetCDF files "
        f"using {max_workers} workers..."
    )

    results = {}

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_path = {
            executor.submit(extract_metadata_from_netcdf, path): path
            for path in file_paths
        }

        with tqdm(
            total=total_files,
            desc="Extracting metadata",
            unit="file"
        ) as pbar:
            for future in as_completed(future_to_path):
                result = future.result()
                results[result['path']] = result
                pbar.update(1)

    expanded_rows = []

    for _, row in catalogue.iterrows():
        path = row['path']
        metadata = results.get(
            path,
            {
                'dates': [],
                'dimensions': None
            }
        )

        dates = metadata.get('dates', [])
        dimensions = metadata.get('dimensions')

        if dates:
            for date in dates:
                new_row = row.to_dict()
                new_row['date'] = date
                new_row['dimensions'] = dimensions
                expanded_rows.append(new_row)
        else:
            new_row = row.to_dict()
            new_row['date'] = None
            new_row['dimensions'] = dimensions
            expanded_rows.append(new_row)

    expanded_df = pd.DataFrame(expanded_rows)

    successful = expanded_df['date'].notna().sum()

    logger.info(
        f"Expanded catalogue: {len(catalogue)} files -> "
        f"{len(expanded_df)} rows ({successful} with dates)"
    )

    return expanded_df

def compute_stats_for_array(data: np.ndarray) -> Optional[Dict[str, float]]:

    data = np.asarray(data, dtype=np.float64).ravel()
    data = data[np.isfinite(data)]

    if data.size == 0:
        return None

    stats_dict = {
        "min": float(np.min(data)),
        "max": float(np.max(data)),
        "mean": float(np.mean(data)),
        "std": float(np.std(data)),
        "median": float(np.median(data)),
        "skewness": float(scipy_stats.skew(data, nan_policy="omit")),
    }

    quantile_values = np.quantile(data, QUANTILES)

    for q, val in zip(QUANTILES, quantile_values):
        stats_dict[f"q{q:.2f}"] = float(val)

    return stats_dict


def extract_stats_from_file_group(
    args: Tuple[str, str, List[str]]
) -> List[Dict[str, Any]]:
    
    file_path, variable_id, date_list = args
    results = []

    try:
        with xr.open_dataset(file_path, decode_times=True) as ds:

            if variable_id not in ds.data_vars:
                for date_str in date_list:
                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": None,
                        "error": f"{variable_id} not found in file"
                    })
                return results

            da = ds[variable_id]

            if "time" in da.dims:
                times = pd.to_datetime(ds["time"].values)
                time_dates = np.array([
                    pd.Timestamp(t).strftime("%Y-%m-%d")
                    for t in times
                ])

                for date_str in date_list:
                    date_str = pd.Timestamp(date_str).strftime("%Y-%m-%d")

                    idx = np.where(time_dates == date_str)[0]

                    if len(idx) == 0:
                        results.append({
                            "path": file_path,
                            "variable_id": variable_id,
                            "date": date_str,
                            "stats": None,
                            "error": "date not found in file"
                        })
                        continue

                    data_slice = da.isel(time=idx).values
                    stats = compute_stats_for_array(data_slice)

                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": stats,
                        "error": None
                    })

            else:
                data_slice = da.values
                stats = compute_stats_for_array(data_slice)

                for date_str in date_list:
                    date_str = pd.Timestamp(date_str).strftime("%Y-%m-%d")

                    results.append({
                        "path": file_path,
                        "variable_id": variable_id,
                        "date": date_str,
                        "stats": stats,
                        "error": None
                    })

    except Exception as e:
        for date_str in date_list:
            results.append({
                "path": file_path,
                "variable_id": variable_id,
                "date": pd.Timestamp(date_str).strftime("%Y-%m-%d"),
                "stats": None,
                "error": str(e)
            })

    return results


def extract_stats_concurrent(
    catalogue: pd.DataFrame,
    expand_stats: bool = True,
    max_workers = None 
) -> pd.DataFrame:

    if max_workers is None:
        max_workers = min(8, os.cpu_count() or 1)
    
    required_cols = {"path", "variable_id", "date"}
    missing_cols = required_cols - set(catalogue.columns)

    if missing_cols:
        raise ValueError(f"catalogue is missing columns: {missing_cols}")

    catalogue = catalogue.copy()

    # Standardise date format
    catalogue["date"] = pd.to_datetime(
        catalogue["date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    # Remove rows without valid dates
    catalogue_valid = catalogue.dropna(subset=["date"]).copy()

    # Group by file and variable so each NetCDF file is opened once
    grouped = (
        catalogue_valid
        .groupby(["path", "variable_id"])["date"]
        .apply(lambda x: sorted(x.dropna().unique().tolist()))
        .reset_index()
    )

    tasks = [
        (row["path"], row["variable_id"], row["date"])
        for _, row in grouped.iterrows()
    ]

    total_tasks = len(tasks)

    logger.info(
        f"Extracting stats from {total_tasks} file-variable groups "
        f"using {max_workers} workers..."
    )

    all_results = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(extract_stats_from_file_group, task): task
            for task in tasks
        }

        with tqdm(
            total=total_tasks,
            desc="Extracting stats",
            unit="file-var"
        ) as pbar:
            for future in as_completed(future_to_task):
                result_list = future.result()
                all_results.extend(result_list)
                pbar.update(1)

    results_df = pd.DataFrame(all_results)

    # Merge stats back onto original BARRA_DAILY catalogue
    output = catalogue.merge(
        results_df,
        on=["path", "variable_id", "date"],
        how="left"
    )

    successful = output["stats"].notna().sum()

    logger.info(
        f"Stats extracted: {successful}/{len(output)} rows"
    )

    if expand_stats:
        stats_expanded = pd.json_normalize(output["stats"])

        output = pd.concat(
            [
                output.drop(columns=["stats"]),
                stats_expanded
            ],
            axis=1
        )

    return output
    

In [4]:
%%time
BARRA_DAILY = extract_dates_concurrent(
    catalogue=BARRA,
    max_workers=MAX_WORKERS
)
BARRA_DAILY.groupby('variable_id')['date'].agg(['min', 'max', 'count'])


2026-06-16 14:15:27 | INFO    | 2695630142:extract_dates_concurrent:L63 - Extracting metadata from 2120 NetCDF files using 48 workers...
Extracting metadata: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2120/2120 [00:02<00:00, 1027.97file/s]
2026-06-16 14:15:31 | INFO    | 2695630142:extract_dates_concurrent:L117 - Expanded catalogue: 2120 files -> 64520 rows (64520 with dates)


CPU times: user 1.06 s, sys: 835 ms, total: 1.9 s
Wall time: 3.63 s


,min,max,count
variable_id,,,
psl,1990-01-01,2025-04-30,12904
ua100m,1990-01-01,2025-04-30,12904
va100m,1990-01-01,2025-04-30,12904
zg500,1990-01-01,2025-04-30,12904
zmla,1990-01-01,2025-04-30,12904


In [ ]:
%%time
BARRA_DAILY_STATS = extract_stats_concurrent(
    catalogue=BARRA_DAILY,
    expand_stats=True,
    max_workers=MAX_WORKERS,
)


2026-06-16 14:15:33 | INFO    | 2695630142:extract_stats_concurrent:L274 - Extracting stats from 2120 file-variable groups using 48 workers...
Extracting stats:  12%|█████████████▎                                                                                                     | 245/2120 [01:40<33:35,  1.08s/file-var]